# 0. 核心优化

DAPO主要是对GRPO的相关改进，主要体现在以下几个方面：

## 去除 KL 散度

KL 散度用于约束模型同初始模型不会显著偏离，但是在训练 long-CoT reasoning model 时，模型分布会显著偏离初始模型，所以去掉KL散度的约束。

## Clip-Higher

传统 Clip：
$$\text{clip}\left(\frac{\pi(\theta)}{\pi_{old}(\theta)}, 1-\varepsilon, 1+\varepsilon\right)$$

故 $pi(\theta)$ 的范围是 $(1 - \varepsilon) \pi_{old} \leq \pi(\theta) \leq (1 + \varepsilon) \pi_{old}$，否则会发生裁减，所以 $pi(\theta)$ 的更新范围会随着 $\pi_{old}$ 变化而变化。

上界裁剪会限制策略的探索能力，导致 熵崩塌 （policy entropy 迅速下降，模型输出趋同）。

DAPO 将上下界裁剪范围 解耦 —— 增大上界 $\epsilon_{upper}$，为低概率的"探索型 token"提供更大提升空间，同时保持较小的下界 $\epsilon_{lower}$。这有效提升了策略多样性，避免熵崩塌。


$$J_{DAPO}(\theta) = \mathbb{E}_{(q,a) \sim D, \{o_i\}_{i=1}^G \sim o_i}^{} \left[
\frac{1}{\sum_{i=1}^G |o_i|} \sum_{i=1}^G \sum_{t=1}^{|o_i|} 
\min\left(r_{i,t}(\theta) \hat{A}_{i,t}, \; 
\mathrm{clip}\left(r_{i,t}(\theta), 1 - \boldsymbol{\varepsilon_{lower}}, 1 + \boldsymbol{\varepsilon_{upper}}\right) \hat{A}_{i,t}\right)
\right]$$


## 动态采样

对于 GRPO 而言，如果采样答案全对或者全错，那么梯度会变成0，模型不会更新参数，合理的办法是**去掉这些全对或者全错的采样**，也就是 DRPO 中动态采样的想法，

具体的，动态采样就是使得采样的答案不会出现全对或者全错的情况，从而避免了梯度为 0。

$$U_{DAPO}(\theta) = \mathbb{E}_{(q,a) \sim D_{1},|q|=\pi_{old}} \left[ \frac{1}{\sum_{i=1}^{G}|o_i|} \sum_{i=1}^{G} \sum_{t=1}^{|o_i|} \min\left( r_{i,t}(\theta) \hat{A}_{i,t}, \text{clip}\left( r_{i,t}(\theta), 1 - \epsilon_{low}, 1 + \epsilon_{high} \right) \hat{A}_{i,t} \right) \right]$$
$$\text{s.t. } \mathbf{0 < \left| \{ o_i \ | \ is\_equivalent(a, o_i) \} \right| < G}.$$


## 逐 token 级策略梯度损失

对于GRPO而言，本身的目标函数对于答案具有偏向性：**对于答案正确的，GRPO偏向于选择答案长度较短的回复，而对于答案错误的，GRPO偏向于让模型生成更长的回复。**

更详细的说：

在 GRPO 的目标函数中：
$$J_{GRPO}(\theta) = \mathbb{E} \frac{1}{G} \sum_{i=1}^{G} \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \min\left(r_{i,t}(\theta) \hat{A}_i, \text{clip}(\cdots) \hat{A}_i\right)$$
每个回答的最终贡献是 $\frac{1}{G}$， 与长度无关 。


回答正确时（$\hat{A}_i > 0$）：

| 回答类型 | 长度 $l$ | 每 token 梯度 | 强度 |
| :--- | :--- | :--- | :--- |
| 短回答 | $l = 100$ | $G \cdot \frac{\hat{A}_i}{100}$ | 强（50倍） |
| 长回答 | $l = 5000$ | $G \cdot \frac{\hat{A}_i}{5000}$ | 弱 |

短正确回答的每个 token 被更强地"奖励"，模型学会：**正确时尽量写短**。

回答错误时（$\hat{A}_i < 0$）：

| 回答类型 | 长度 $l$ | 每 token 梯度 | 惩罚强度 |
| :--- | :--- | :--- | :--- |
| 短回答 | $l = 100$ | $G \cdot \frac{\hat{A}_i}{100}$ | 强（50倍） |
| 长回答 | $l = 5000$ | $G \cdot \frac{\hat{A}_i}{5000}$ | 弱 |

长错误回答被稀释了惩罚信号，模型学会： 错了就写长一点来"躲"惩罚 。

DAPO 的 token 级损失：
$$J^{\mathrm{DAPO}}(\theta)=\mathbb{E}\left[\frac{1}{\sum_{i=1}^{G}|o_i|}\sum_{i=1}^{G}\sum_{t=1}^{|o_i|}\min\bigl(r_{i,t}(\theta)\hat{A}_{i,t},\;\mathrm{clip}\bigl(r_{i,t}(\theta),\,1-\varepsilon_{\mathrm{low}},\,1+\varepsilon_{\mathrm{high}}\bigr)\hat{A}_{i,t}\bigr)\right]$$

核心变化只有一处：
$$\frac{1}{G}\cdot\frac{1}{|o_i|} \;\longrightarrow\; \frac{1}{\sum_{i=1}^{G} |o_i|}$$

| 特性 | GRPO（样本级） | DAPO（Token 级） |
|------|---------------|-----------------|
| 归一化分母 | $G \cdot l_i$（分两步） | $\sum l_i$（全局总 token 数） |
| 回答权重 | 每个回答 = $1/G$ | 每个回答 $\propto$ 其长度 |
| Token 权重 | $\propto 1/l_i$（长回答的 token 被稀释） | 所有 token 完全平等 |


## 超长奖励塑形

在原始的奖励函数上增加一个关于长度的奖励，从而避免过长后截断导致模型无法得到奖励的情形。

$$R_{\mathrm{length}}(y)=\begin{cases}
0, & |y|\le L_{\max}-L_{\mathrm{cache}} \\
\dfrac{\left(L_{\max}-L_{\mathrm{cache}}\right)-|y|}{L_{\mathrm{cache}}}, & L_{\max}-L_{\mathrm{cache}}<|y|\le L_{\max} \\
-1, & L_{\max}<|y|
\end{cases}$$

- $L_{\text{max}}$：最大生成长度硬上限。模型单次推理最多生成的 token 数，超过这个长度回答会被强制截断。在 DAPO 论文中设为 20480 token。
- $L_{\text{cache}}$：缓冲惩罚区间的宽度。在 $L_{\text{max}}$ 之前留出的一个"预警带"，在这个区间内长度惩罚从 0 线性递增到 -1。在 DAPO 论文中设为 4096 token。

直观理解：$L_{\text{max}}$ 是天花板（撞到就断），$L_{\text{cache}}$ 是"快撞到了"的报警距离。报警距离越大（$L_{\text{cache}}$ 越大），模型越早收到"该收尾了"的信号，惩罚增长越平缓。


# 1. 补充

**lip-Higher 把 $\varepsilon_{\text{low}}$ 和 $\varepsilon_{\text{high}}$ 解耦，为什么非对称？能不能反过来 $\varepsilon_{\text{low}} > \varepsilon_{\text{high}}$？**

核心是要打破探索瓶颈：低概率 token 的 $\mathcal{L}_{i,t}(\theta) \gg 1$，需要大上界才能不被 clip 掉提升空间。反过来 $\varepsilon_{\text{low}} > \varepsilon_{\text{high}}$ 会强烈抑制探索（稍微走出舒适区就被 clip），等价于鼓励策略回退。可以要求推导出 $\mathcal{L}_{i,t}(\theta) > 1 + \varepsilon$ 时被截断的含义。


# 2. Reference